# Style Transfer (Using LoRA)

### Get the Image Dataset 

This is a dataset of ~1800 Studio Ghibli style hand-drawn images, which we will use to give our comics the same hand-drawn effect.
Make sure to generate captions for these images using `image_captioning.ipynb`

In [ ]:
# Get the Studio Ghibli Style Images Dataset... and Unzip it

!wget https://github.com/TachibanaYoshino/AnimeGANv2/releases/download/1.0/Hayao.tar.gz
!tar -xzvf Hayao.tar.gz

### Imports and Initial Setup

In [ ]:
# Manage Hex Cloud GPU Usage...
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"  

In [ ]:
# Import Libraries

import csv
import lightning.pytorch as pl
import numpy             as np
import torch
import torch.nn          as nn
import torch.optim       as optim

from diffusers           import AutoPipelineForImage2Image, DPMSolverMultistepScheduler, StableDiffusionPipeline
from peft                import get_peft_model, LoraConfig, PeftModel
from PIL                 import Image
from torch.utils.data    import DataLoader, Dataset 
from torchvision         import transforms


### LoRA Setup

In [ ]:
CAPTIONS_FILE = "image_captions.csv"

# Dataset to handle the image-caption pairs...
class GhibliDataset(Dataset):
    def __init__(self, dataset_path):
        self.dataset_path = dataset_path

        self.image_paths     = []
        for filename in os.listdir(self.dataset_path):
            if filename.lower().endswith(("jpg", "png", "jpeg")): # Check for file image format
                self.image_paths.append(os.path.join(self.dataset_path, filename))
        self.image_paths = sorted(self.image_paths)

        with open(CAPTIONS_FILE, mode='r', encoding='utf-8') as csv_file:
            reader                   = csv.DictReader(csv_file)
            self.filename_to_caption = {row['filename'].strip(): row['caption'].strip() for row in reader}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        
        image = Image.open(image_path).convert("RGB")
        
        # Resize the image to 512x512
        image = transforms.Resize(512, interpolation=transforms.InterpolationMode.BILINEAR)(image)
        image = transforms.RandomCrop(512)(image)
        # Convert the image to a tensor
        image = transforms.ToTensor()(image)
        # Normalize the image with mean and std of 0.5
        image = transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])(image)

        caption = self.filename_to_caption.get(image_path, "A Ghibli style scene")
        return image, caption


In [ ]:
# Class to handle training the LoRA

class LoRATrainer(pl.LightningModule):
    def __init__(self, pipeline, lr, train_strength=0.5):
        super().__init__() # Initialize parent class
        
        self.unet           = pipeline.unet
        self.vae            = pipeline.vae
        self.tokenizer      = pipeline.tokenizer
        self.text_encoder   = pipeline.text_encoder
        self.scheduler      = pipeline.scheduler
        self.train_strength = train_strength
        self.lr             = lr

        self.prefixes = [
            "A Ghibli style image showing ",
            "A Ghibli style painting depicting ",
            "A Ghibli style scene of "
        ]

    def training_step(self, batch, batch_idx):
        images, captions = batch
        
        # Add a prefix to the captions to represent our style...
        captions      = [np.random.choice(self.prefixes) + caption for caption in captions]

        # Tokenize captions
        tokens        = self.tokenizer(captions, return_tensors='pt', padding=True, truncation=True).input_ids.to(self.device)
        
        # Encode images to latents
        latents       = self.vae.encode(images).latent_dist.sample() * 0.18215
        
        # Add noise
        noise         = torch.randn_like(latents)
        timesteps     = torch.randint(0, int(1000 * self.train_strength), (images.size(0),), device=self.device).long()
        noisy_latents = self.scheduler.add_noise(latents, noise, timesteps)
        
        # Predict noise
        noise_pred    = self.unet(noisy_latents, timesteps, encoder_hidden_states=self.text_encoder(tokens)[0])['sample']
        
        # Compute MSE loss
        loss          = torch.nn.functional.mse_loss(noise_pred, noise)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.unet.parameters(), lr=self.lr)


### Training

In [ ]:
# Hyperparameters

BASE_MODEL     = "runwayml/stable-diffusion-v1-5"
DATASET_PATH   = "Hayao/style"
LORA_RANK      = 4
LORA_ALPHA     = 8
LORA_DROPOUT   = 0.1
TRAIN_STRENGTH = 0.5
LEARNING_RATE  = 1e-6
BATCH_SIZE     = 4
GRAD_ACC_STEPS = 4
NUM_EPOCHS     = 100
OUTPUT_DIR     = "ghibli_lora"

In [ ]:

torch.cuda.empty_cache()

pipeline                = StableDiffusionPipeline.from_pretrained(BASE_MODEL)
pipeline.scheduler      = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
pipeline.safety_checker = None  # Disable for creative outputs


# Inject LoRA using the PEFT Library
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    init_lora_weights='gaussian',
    target_modules=['to_k', 'to_q', 'to_v', 'to_out.0', 'ResnetBlock2D', 'CrossAttention'],
    lora_dropout=LORA_DROPOUT,
    bias='none'
)
pipeline.unet = get_peft_model(pipeline.unet, lora_config)

# Prepare Dataloader
dataset    = GhibliDataset(DATASET_PATH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

# Train!
model = LoRATrainer(pipeline=pipeline, lr=LEARNING_RATE, train_strength=TRAIN_STRENGTH)
trainer = pl.Trainer(
    max_epochs=NUM_EPOCHS,
    precision='16-mixed',
    accumulate_grad_batches=GRAD_ACC_STEPS
)
trainer.fit(model, train_dataloaders=dataloader)


# Save
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.unet.save_pretrained(OUTPUT_DIR)

### Test Inference

In [ ]:
# Text-to-Image Generation

PROMPT = "A girl with golden hair walks into the forest."

pipeline                = StableDiffusionPipeline.from_pretrained(BASE_MODEL)
pipeline.scheduler      = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
pipeline.unet           = PeftModel.from_pretrained(pipeline.unet, OUTPUT_DIR)
pipeline.safety_checker = None
pipeline                = pipeline.to('cuda')

# Add "A Ghibli style scene of" to trigger generation in the trained art-style...
ghibli_prompt = f"A Ghibli style scene of {PROMPT}"

image = pipeline(prompt=ghibli_prompt,
                 num_inference_steps=40,
                 guidance_scale=8
                ).images[0]

display(image)

In [ ]:
# Image-to-Image Style Transfer

IMAGE_PATH = "image.png"

# Load Stable Diffusion pipeline...
pipeline                    = AutoPipelineForImage2Image.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
pipeline.scheduler          = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
pipeline.safety_checker     = None
pipeline                    = pipeline.to(device)


# Load IP-Adapter
pipeline.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sd15.bin"
)
pipeline.set_ip_adapter_scale(0.5)

# Load LoRA
pipeline.unet = PeftModel.from_pretrained(pipeline.unet, OUTPUT_DIR)
pipeline.unet.eval()

# Load and Process Image
image = Image.open(image_path).convert("RGB")

# Size 512X512 for SD...
image_sd = transforms.Resize((512, 512), interpolation=transforms.InterpolationMode.BILINEAR)(image)
image_sd = transforms.CenterCrop(512)(image)
# Size 224x224 for the IP-Adapter...
image_ip = transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR)(image)
image_ip = transforms.CenterCrop(224)(image_ip)
image_ip = transforms.ToTensor()(image_ip).unsqueeze(0).to("cuda")


# Run the style transfer...
with torch.autocast("cuda"):
    image = pipeline(
        prompt="A Ghibli style scene", # Prompt contains "Ghibli style" so that generated image is of that style
        image=image_sd,
        ip_adapter_image=image_ip,
        num_inference_steps=40,
        strength=0.95,
        guidance_scale=8.5,
        negative_prompt="blurry face"
    ).images[0]

display(image)